# CDC-7 — CDC → Spark → Iceberg upsert pipeline

**Break → Detect → Fix → Prove.** The payoff of the whole track: consume the Debezium change
stream in Spark and **`MERGE` it into an Iceberg table that mirrors the Postgres source** —
applying inserts/updates as upserts and deletes as removals, **idempotently** (replays don't
double-apply, because we keep the latest change per key by **LSN**).

**Prerequisite:** `make cdc-up`. **Laptop-safe:** 15-row table, bounded reads, full teardown.

> **Connect note:** we read the topic with a **batch** `spark.read` and run **one MERGE** — fully
> deterministic for this lab. The production shape is a `readStream` + `foreachBatch(merge_fn)` +
> `trigger(availableNow=True)`; that snippet is in §7. We use `snapshot.mode="always"` so the demo
> re-runs cleanly every time (production uses `initial` — see CDC-3).

In [ ]:
from common import cdc_helpers as cdc
from common.kafka_helpers import SPARK_BOOTSTRAP
from common.spark_session import spark
from pyspark.sql import functions as F, Window
from pyspark.sql.types import StructType, StructField, StringType, LongType, DoubleType
import time

NAME, TABLE = "cdc7-orders", "cdc7_orders"
TOPIC = cdc.topic_name(TABLE)
MIRROR = "iceberg_catalog.default.cdc7_orders_mirror"

cdc.teardown(NAME, TABLE)                       # clean slate (idempotent re-run)
cdc.seed_orders(TABLE, n=15)
print("seeded 15 rows into public.cdc7_orders; CDC topic:", TOPIC)

## 1. Bring up the connector and snapshot the source

`snapshot.mode="always"` re-snapshots on every start, so this lab is deterministic on re-run.

In [ ]:
cdc.register_connector(NAME, cdc.debezium_pg_config(NAME, TABLE, snapshot_mode="always"))
print("connector:", cdc.wait_for_connector(NAME, timeout=60))
# wait until the 15 snapshot rows have landed on the topic before Spark subscribes
from common.kafka_helpers import topic_end_offsets
for _ in range(20):
    try: off = topic_end_offsets(TOPIC)
    except Exception: off = {}
    if isinstance(off, dict) and sum(off.values()) >= 15: break
    time.sleep(2)
print("snapshot landed; topic end offsets:", off)

## 2. Apply changes in Postgres — an update, a delete, and an insert

In [ ]:
cdc.pg_exec(f"UPDATE public.{TABLE} SET status='PAID', amount=999.00 WHERE id=1")
cdc.pg_exec(f"DELETE FROM public.{TABLE} WHERE id=2")
cdc.pg_exec(f"INSERT INTO public.{TABLE}(id,customer,amount,status) VALUES (200,'newbie',12.50,'NEW')")
time.sleep(6)   # let Debezium decode the WAL changes onto the topic
print("applied U(id=1), D(id=2), I(id=200)")

## 3. Read the CDC stream in Spark & parse the Debezium envelope

The value is JSON: `before` / `after` / `op` (`r`,`c`,`u`,`d`) / `ts_ms` / `source.lsn`. We pull the
key fields, taking the row image from `after` (or `before` for deletes) and the **LSN** for ordering.

In [ ]:
after = StructType([StructField("id", LongType()), StructField("customer", StringType()),
                    StructField("amount", DoubleType()), StructField("status", StringType()),
                    StructField("updated", LongType())])
source = StructType([StructField("lsn", LongType()), StructField("ts_ms", LongType())])
envelope = StructType([StructField("before", after), StructField("after", after),
                       StructField("op", StringType()), StructField("ts_ms", LongType()),
                       StructField("source", source)])

raw = (spark.read.format("kafka")
       .option("kafka.bootstrap.servers", SPARK_BOOTSTRAP)
       .option("subscribe", TOPIC).option("startingOffsets", "earliest").load())
evt = raw.select(F.from_json(F.col("value").cast("string"), envelope).alias("e")).select("e.*")
print("events on the topic by op:")
evt.groupBy("op").count().orderBy("op").show()

## 4. Dedup to the latest change per key (by LSN) — idempotent upsert prep

A key may have several events (snapshot `r`, then `u`/`d`). We keep **one row per `id`** — the
highest `source.lsn` — so the MERGE applies the final state and re-running is a no-op. Tombstones
(`op=null`) are dropped; the `d` event already carries the delete intent.

In [ ]:
changes = (evt.filter(F.col("op").isNotNull())
           .select("op", F.col("source.lsn").alias("lsn"),
                   F.coalesce("after.id", "before.id").alias("id"),
                   F.col("after.customer").alias("customer"),
                   F.col("after.amount").alias("amount"),
                   F.col("after.status").alias("status"),
                   F.col("after.updated").alias("updated")))
w = Window.partitionBy("id").orderBy(F.col("lsn").desc_nulls_last())
latest = changes.withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")
print("deduped change rows (one per id):", latest.count())

## 5. MERGE into the Iceberg mirror

`op='d'` → `DELETE`; any other op → upsert. Iceberg's `MERGE INTO` does it in one statement.

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {MIRROR}")
spark.sql(f"""CREATE TABLE {MIRROR} (
    id BIGINT, customer STRING, amount DOUBLE, status STRING, updated BIGINT
) USING iceberg""")
latest.createOrReplaceTempView("cdc7_changes")

MERGE = f"""MERGE INTO {MIRROR} t USING cdc7_changes s ON t.id = s.id
WHEN MATCHED AND s.op = 'd' THEN DELETE
WHEN MATCHED THEN UPDATE SET t.customer=s.customer, t.amount=s.amount, t.status=s.status, t.updated=s.updated
WHEN NOT MATCHED AND s.op <> 'd' THEN INSERT (id, customer, amount, status, updated)
                                  VALUES (s.id, s.customer, s.amount, s.status, s.updated)"""
spark.sql(MERGE)
print("merged.")

## 6. Prove — the Iceberg mirror matches Postgres

In [ ]:
mirror = {r["id"]: r["status"] for r in spark.sql(f"SELECT id, status FROM {MIRROR}").collect()}
pg = {r[0]: r[1] for r in cdc.pg_exec(f"SELECT id, status FROM public.{TABLE}", fetch=True)}
print(f"mirror rows: {len(mirror)} | postgres rows: {len(pg)} | MATCH: {mirror == pg}")
print(f"id=1 status (updated→PAID): {mirror.get(1)}")
print(f"id=2 removed by delete:     {2 not in mirror}")
print(f"id=200 inserted:            {200 in mirror}")
assert mirror == pg, "mirror should equal the Postgres source"
print("✓ Iceberg mirror is a faithful, up-to-date replica of the Postgres table.")

## 7. Idempotency — re-applying the same changes changes nothing

Because we keyed the MERGE on `id` and kept the latest-by-LSN row, replaying the **same** batch is a
**no-op** — the basis of *effectively-once* delivery on top of at-least-once CDC (see CDC-9).

In [ ]:
spark.sql(MERGE)   # replay
mirror2 = {r["id"]: r["status"] for r in spark.sql(f"SELECT id, status FROM {MIRROR}").collect()}
print("re-MERGE identical:", mirror2 == mirror, "| rows:", len(mirror2))

## 8. Diagnose & "in real production…"

- **Productionize as a stream.** Swap the batch read for a continuous one with `foreachBatch` (the
  same dedup + MERGE per micro-batch), bounded here with `availableNow`:

  ```python
  def upsert(batch_df, batch_id):
      batch_df.createOrReplaceTempView("cdc7_changes")
      batch_df.sparkSession.sql(MERGE)   # dedup-latest-per-key first, then MERGE

  (spark.readStream.format("kafka")
       .option("kafka.bootstrap.servers", SPARK_BOOTSTRAP)
       .option("subscribe", TOPIC).option("startingOffsets", "earliest").load()
       .select(F.from_json(F.col("value").cast("string"), envelope).alias("e")).select("e.*")
       .writeStream.foreachBatch(upsert)
       .option("checkpointLocation", ".tmp/checkpoint_cdc7")
       .trigger(availableNow=True).start().awaitTermination())
  ```

- **Idempotent sink + LSN guard** is what makes the at-least-once CDC stream safe: dedup by key,
  keep the max-LSN row, and a redelivered batch can't double-apply. The Spark **checkpoint** (STR-2)
  + `read_committed` (KAF-5) complete the **effectively-once** story (CDC-9).
- **Deletes** need the primary key in the event — guaranteed here by the PK; capturing old non-key
  values needs `REPLICA IDENTITY FULL` (CDC-6). Schema drift → evolve the Iceberg table (CDC-8 / LAK-6).

## Teardown

In [ ]:
spark.sql(f"DROP TABLE IF EXISTS {MIRROR}")
cdc.teardown(NAME, TABLE)
print("dropped mirror + connector + slot + table + topic. `make clean` clears the rest.")